# Embedding benchmark: Qwen3-Embedding 0.6B vs 4B

Generated by `scripts/build_embed_benchmark_notebook.py`. Edit the generator, not this file.

- Kaggle T4 x2: Internet ON, attach `boinhbo/medxir-corpus`, Run All.
- Kaggle RTX Pro 6000: Internet OFF, attach `boinhbo/medxir-corpus` and `boinhbo/qwen3-embedding`, Run All.

Models are read from `/kaggle/input` when attached, otherwise downloaded from Hugging Face.


In [ ]:
import glob
import os

# Parameters: change here only.
MODELS = ['Qwen/Qwen3-Embedding-0.6B', 'Qwen/Qwen3-Embedding-4B']
N_CHUNKS = 100
BATCH_SIZES = [8, 16, 32]
MAX_LENGTH = 1024
REPEATS = 3
SEED = 0
DTYPE_NAME = 'auto'      # auto: bfloat16 on compute capability >= 8 (RTX Pro 6000), else float16 (T4)
ROW_GROUPS_PER_LANG = 4  # parquet row groups read per language
DOCS_PER_LANG = 500      # documents chunked per language (sample pool and token statistics)
CORPUS_CHUNKS = {'vi': 2_190_000, 'en': 252_000, 'zh': 264_000}  # full-corpus chunk estimate
WORK = os.environ.get('WORK_DIR', '/kaggle/working')
DATA_ROOT = os.environ.get('DATA_ROOT')  # None on Kaggle: registry.parquet is searched under /kaggle/input


def local_model_dir(repo):
    """Folder of the attached offline model (dataset boinhbo/qwen3-embedding), or None."""
    name = repo.split('/')[-1]
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}/config.json', recursive=True))
    return os.path.dirname(hits[0]) if hits else None


MODEL_SOURCE = {repo: local_model_dir(repo) or repo for repo in MODELS}
if all(src.startswith('/') for src in MODEL_SOURCE.values()):
    os.environ['HF_HUB_OFFLINE'] = '1'
os.makedirs(WORK, exist_ok=True)
for repo, src in MODEL_SOURCE.items():
    print(f'{repo} -> {src}')


## 1. Environment


In [ ]:
import importlib.metadata as md
import re
import socket
import subprocess
import sys

print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout)
for pkg in ['torch', 'transformers', 'sentence-transformers', 'accelerate']:
    try:
        print(f'{pkg:24s}', md.version(pkg))
    except md.PackageNotFoundError:
        print(f'{pkg:24s} not installed')


def online():
    try:
        socket.create_connection(('pypi.org', 443), timeout=3).close()
        return True
    except OSError:
        return False


def version_tuple(name):
    return tuple(int(part) for part in re.findall(r'\d+', md.version(name))[:2])


# Qwen3-Embedding needs transformers>=4.51. Upgrade only when online and too old.
if online() and version_tuple('transformers') < (4, 51):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.51'], check=True)
    print('transformers upgraded: restart the kernel and Run All if an import fails')


## 2. Chunking code (scripts/chunking.py, inlined)

Same functions as the corpus build. Only the `__main__` guard is left out, so the notebook does not run argparse.


In [ ]:
"""Paragraph-first chunking with exact source offsets.

Every chunk is text[start:end] of the exported document, so chunk_text is always
a verbatim extract (SUBMISSION.md). Size is counted in words, or in characters/2
for Chinese. The scorer divides overlap by the *reference* chunk length, so a
chunk shorter than 40% of the reference can never match; err on the large side.

    uv run python scripts/chunking.py stats artifacts/corpus_v1/en.parquet --limits 120 250 400
"""
import argparse
import re
from dataclasses import dataclass

PARAGRAPH = re.compile(r'\n\s*\n')
SENTENCE_END = re.compile(r'(?<=[.!?。！？；;])\s+|(?<=[。！？；])')


@dataclass
class Chunk:
    order: int
    start: int
    end: int
    text: str


def size(text, lang):
    return len(text) // 2 if lang == 'zh' else len(text.split())


def spans(text, pattern, start=0, end=None):
    """Non-empty (start, end) spans of text[start:end] split by pattern, whitespace-trimmed."""
    end = len(text) if end is None else end
    cursor = start
    for match in list(pattern.finditer(text, start, end)) + [None]:
        stop = match.start() if match else end
        piece = text[cursor:stop]
        if piece.strip():
            left = cursor + len(piece) - len(piece.lstrip())
            right = cursor + len(piece.rstrip())
            yield left, right
        if match:
            cursor = match.end()


def sentences(text, start, end, lang, limit):
    """Sentence spans; a sentence above the limit (tables, lists without punctuation)
    is cut at line breaks, then hard-cut at whitespace (or every 2*limit chars for zh)."""
    for s_start, s_end in spans(text, SENTENCE_END, start, end):
        if size(text[s_start:s_end], lang) <= limit:
            yield s_start, s_end
            continue
        for l_start, l_end in spans(text, re.compile(r'\n'), s_start, s_end):
            if size(text[l_start:l_end], lang) <= limit:
                yield l_start, l_end
            elif lang == 'zh':
                for cut in range(l_start, l_end, 2 * limit):
                    yield cut, min(cut + 2 * limit, l_end)
            else:
                words = [m.span() for m in re.finditer(r'\S+', text[l_start:l_end])]
                for i in range(0, len(words), limit):
                    piece = words[i:i + limit]
                    yield l_start + piece[0][0], l_start + piece[-1][1]


def units(text, lang, limit):
    """Paragraphs, with paragraphs above the limit split into sentence groups."""
    for start, end in spans(text, PARAGRAPH):
        if size(text[start:end], lang) <= limit:
            yield start, end
            continue
        group = None
        for s_start, s_end in sentences(text, start, end, lang, limit):
            if group and size(text[group[0]:s_end], lang) > limit:
                yield group
                group = None
            group = (group[0], s_end) if group else (s_start, s_end)
        if group:
            yield group


def is_heading(text, lang):
    """Short line without closing punctuation: a section title or list label."""
    text = text.strip()
    short = len(text) <= 30 if lang == 'zh' else len(text.split()) <= 12
    return short and not text.endswith(('.', '!', '?', ':', ';', '。', '！', '？', '：', '；', '"', '”', ')'))


def chunk_document(text, lang, limit=350, minimum=40, overlap=50):
    """Greedily merge adjacent units up to `limit`; fold a tiny tail into the previous chunk.

    Headings never end a chunk: trailing headings move to the next chunk, so a title
    always stays with the text it introduces. A chunk that continues the same section
    also repeats the previous chunk's last whole sentences (up to `overlap` units);
    a chunk opening on a heading starts a new section and gets no overlap.
    """
    groups = []
    for unit in units(text, lang, limit):
        if groups and size(text[groups[-1][0][0]:unit[1]], lang) <= limit:
            groups[-1].append(unit)
            continue
        carried = []
        while groups and len(groups[-1]) > 1 and is_heading(text[groups[-1][-1][0]:groups[-1][-1][1]], lang):
            carried.insert(0, groups[-1].pop())
        groups.append(carried + [unit])
    merged = [(group[0][0], group[-1][1]) for group in groups]
    if len(merged) > 1 and size(text[merged[-1][0]:merged[-1][1]], lang) < minimum:
        tail = merged.pop()
        merged[-1] = (merged[-1][0], tail[1])
    chunks = []
    for order, (start, end) in enumerate(merged):
        if order and overlap and not is_heading(text[start:end].split('\n', 1)[0], lang):
            previous_start, previous_end = merged[order - 1]
            for s_start, _ in reversed(list(spans(text, SENTENCE_END, previous_start, previous_end))):
                if size(text[s_start:previous_end], lang) > overlap:
                    break
                start = s_start
        chunks.append(Chunk(order, start, end, text[start:end]))
    return chunks


def sample_rows(path, sample):
    """(lang, text) of a random sample, read one row group at a time: take() on the whole
    text column overflows the 2 GB string offsets of the 1 GB vi corpus."""
    import numpy as np
    import pyarrow.parquet as pq

    file = pq.ParquetFile(path)
    total = file.metadata.num_rows
    wanted = None
    if sample and total > sample:
        wanted = set(np.random.default_rng(0).choice(total, sample, replace=False).tolist())
    rows, offset = [], 0
    for group in range(file.metadata.num_row_groups):
        count = file.metadata.row_group(group).num_rows
        columns = file.read_row_group(group, columns=['lang', 'text']).to_pydict()
        rows += [(columns['lang'][i], columns['text'][i]) for i in range(count)
                 if wanted is None or offset + i in wanted]
        offset += count
    return rows


def stats(path, limits, sample):
    import numpy as np

    rows = sample_rows(path, sample)
    langs, texts = [lang for lang, _ in rows], [text for _, text in rows]
    lengths = np.array([size(t, l) for t, l in zip(texts, langs)])
    unit = 'chars/2' if langs and langs[0] == 'zh' else 'words'
    print(f'{path}: {len(texts):,} documents sampled, length in {unit}')
    print('  document length p10/p50/p90/p99:', *np.percentile(lengths, [10, 50, 90, 99]).astype(int))
    for limit in limits:
        counts, chunk_sizes = [], []
        for text, lang in zip(texts, langs):
            chunks = chunk_document(text, lang, limit)
            counts.append(len(chunks))
            chunk_sizes.extend(size(c.text, lang) for c in chunks)
            assert all(text[c.start:c.end] == c.text for c in chunks)
        scale = len(texts) if not sample else None
        print(f'  limit {limit:>4}: chunks/doc mean {np.mean(counts):.2f}, '
              f'chunk size p50/p90 {int(np.percentile(chunk_sizes, 50))}/{int(np.percentile(chunk_sizes, 90))}, '
              f'single-chunk docs {np.mean(np.array(counts) == 1):.0%}'
              + (f', total {sum(counts):,}' if scale else ''))


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('action', choices=['stats'])
    parser.add_argument('parquet')
    parser.add_argument('--limits', type=int, nargs='+', default=[120, 250, 400])
    parser.add_argument('--sample', type=int, default=20000, help='0 = all documents')
    args = parser.parse_args()
    stats(args.parquet, args.limits, args.sample)


## 3. Sample 100 chunks

Stratified by the corpus chunk share (VI 81, EN 9, ZH 10). The chunk set is written once to
`bench_chunks.parquet`, and both models read that file. The full sample pool (`POOL_DF`) also feeds the token statistics.


In [ ]:
import random

import pandas as pd
import pyarrow.parquet as pq

LANGS = ['vi', 'en', 'zh']
if DATA_ROOT is None:
    hits = glob.glob('/kaggle/input/**/registry.parquet', recursive=True)
    if not hits:
        raise FileNotFoundError('medxir-corpus is not attached')
    DATA_ROOT = os.path.dirname(hits[0])
print('DATA_ROOT:', DATA_ROOT)


def allocate(total, shares):
    """Split total by shares with the largest-remainder method."""
    raw = {key: total * value / sum(shares.values()) for key, value in shares.items()}
    alloc = {key: int(value) for key, value in raw.items()}
    for key in sorted(raw, key=lambda k: raw[k] - alloc[k], reverse=True)[:total - sum(alloc.values())]:
        alloc[key] += 1
    return alloc


quota = allocate(N_CHUNKS, CORPUS_CHUNKS)
print('chunks per language:', quota)

rng = random.Random(SEED)
pool, picked = [], []
for lang in LANGS:
    parts = sorted(glob.glob(f'{DATA_ROOT}/docs/{lang}/*.parquet'))
    groups = [(part, g) for part in parts for g in range(pq.ParquetFile(part).metadata.num_row_groups)]
    rng.shuffle(groups)
    docs = []
    for part, g in groups[:ROW_GROUPS_PER_LANG]:
        docs += pq.ParquetFile(part).read_row_group(g, columns=['doc_id', 'lang', 'url', 'text']).to_pylist()
    docs = [doc for doc in docs if doc['text']]
    lang_pool = []
    for doc in rng.sample(docs, min(DOCS_PER_LANG, len(docs))):
        for chunk in chunk_document(doc['text'], doc['lang']):
            lang_pool.append(dict(doc_id=doc['doc_id'], lang=doc['lang'], url=doc['url'],
                                  order=chunk.order, start=chunk.start, end=chunk.end, text=chunk.text))
    assert len(lang_pool) >= quota[lang], f'{lang}: only {len(lang_pool)} chunks in the pool'
    pool += lang_pool
    picked += rng.sample(lang_pool, quota[lang])

POOL_DF = pd.DataFrame(pool)
bench = pd.DataFrame(picked).sort_values(['lang', 'doc_id', 'order']).reset_index(drop=True)
bench.insert(0, 'chunk_id', [f'{d}-{o}' for d, o in zip(bench.doc_id, bench.order)])
bench['units'] = [size(text, lang) for text, lang in zip(bench.text, bench.lang)]
bench.to_parquet(f'{WORK}/bench_chunks.parquet', index=False)
print(f'saved {len(bench)} chunks to {WORK}/bench_chunks.parquet; pool {len(POOL_DF):,} chunks')
print(bench.groupby('lang').agg(chunks=('chunk_id', 'size'), docs=('doc_id', 'nunique'),
                                units_mean=('units', 'mean'), units_max=('units', 'max')).to_string())


## 4. Token counts per language (each model's tokenizer)

Counted on the sample pool, not on the 100 chunks, so the EN and ZH means are not based on 9 and 10 chunks.
Tokens are truncated to `MAX_LENGTH`, which is what the encoder computes.


In [ ]:
import numpy as np
from transformers import AutoTokenizer

token_rows, TOKEN_MEAN = [], {}
for repo in MODELS:
    tok = AutoTokenizer.from_pretrained(MODEL_SOURCE[repo])
    raw = np.array([len(ids) for ids in tok(POOL_DF.text.tolist())['input_ids']])
    frame = pd.DataFrame({'lang': POOL_DF.lang.values, 'raw': raw, 'tokens': np.minimum(raw, MAX_LENGTH)})
    TOKEN_MEAN[repo] = {}
    for lang, sub in frame.groupby('lang'):
        TOKEN_MEAN[repo][lang] = sub.tokens.mean()
        token_rows.append(dict(model=repo, lang=lang, chunks=len(sub), mean=round(sub.tokens.mean(), 1),
                               p90=sub.tokens.quantile(0.9), max_raw=int(sub.raw.max()),
                               over_max_len=round((sub.raw > MAX_LENGTH).mean(), 4)))
print(pd.DataFrame(token_rows).to_string(index=False))


## 5. Benchmark

fp16 on T4 (no bf16), bf16 on newer GPUs. SDPA attention, no flash-attn 2.
Documents get no instruction. Queries use the model card's instruction format (shown once, not timed).
Tokenization is done before timing. Each timed pass covers the forward and pooling of all 100 chunks, with a warmup and an untimed sanity pass first.


In [ ]:
import gc
import json
import statistics
import time
from concurrent.futures import ThreadPoolExecutor

import torch
import torch.nn.functional as F
from transformers import AutoModel

TASK = 'Given a web search query, retrieve relevant passages that answer the query'
N_GPU = torch.cuda.device_count()
GPU_NAME = torch.cuda.get_device_name(0)
if DTYPE_NAME == 'auto':
    # torch.cuda.is_bf16_supported() can be True on T4 (emulated), so check the compute capability.
    DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
else:
    DTYPE = getattr(torch, DTYPE_NAME)
print('GPUs:', N_GPU, GPU_NAME, '| dtype:', DTYPE, '| attention: sdpa')


def get_detailed_instruct(task, query):
    return f'Instruct: {task}\nQuery:{query}'


def load(src, device):
    start = time.perf_counter()
    tok = AutoTokenizer.from_pretrained(src)
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModel.from_pretrained(src, torch_dtype=DTYPE, attn_implementation='sdpa')
    model = model.to(device).eval()
    torch.cuda.synchronize(device)
    return tok, model, time.perf_counter() - start


def count_tokens(tok, texts):
    return tok(texts, truncation=True, max_length=MAX_LENGTH)['input_ids']


def make_batches(tok, ids, indices, batch_size):
    """Left-padded batches of the given chunk indices, longest first (as sentence-transformers does)."""
    order = sorted(indices, key=lambda i: -len(ids[i]))
    batches = []
    for start in range(0, len(order), batch_size):
        chunk = order[start:start + batch_size]
        enc = tok.pad({'input_ids': [ids[i] for i in chunk]}, return_tensors='pt')
        batches.append(dict(idx=chunk, input_ids=enc['input_ids'], mask=enc['attention_mask'],
                            real=sum(len(ids[i]) for i in chunk), padded=enc['input_ids'].numel()))
    return batches


def last_token_pool(hidden, mask):
    """Last-token pooling from the Qwen3-Embedding model card (handles left and right padding)."""
    if mask[:, -1].sum() == mask.shape[0]:
        return hidden[:, -1]
    lengths = mask.sum(dim=1) - 1
    return hidden[torch.arange(hidden.shape[0], device=hidden.device), lengths]


@torch.inference_mode()
def embed(model, batches, device, keep=False):
    """Forward over the batches; returns {chunk index: L2-normalised vector} when keep is True."""
    out = {}
    for b in batches:
        mask = b['mask'].to(device)
        hidden = model(input_ids=b['input_ids'].to(device), attention_mask=mask, use_cache=False).last_hidden_state
        vec = F.normalize(last_token_pool(hidden, mask).float(), p=2, dim=1)
        if keep:
            for i, v in zip(b['idx'], vec.cpu().numpy()):
                out[i] = v
    return out


def timed(fn, devices):
    for d in devices:
        torch.cuda.synchronize(d)
    start = time.perf_counter()
    fn()
    for d in devices:
        torch.cuda.synchronize(d)
    return time.perf_counter() - start


def summarize(times, real, padded, peak_gb, load_s, vectors, n):
    median = statistics.median(times)
    E = np.stack([vectors[i] for i in range(n)])
    norms = np.linalg.norm(E, axis=1)
    return dict(status='ok', load_s=round(load_s, 1), peak_vram_gb=round(peak_gb, 2), median_s=round(median, 3),
                chunks_per_s=n / median, real_tok_per_s=real / median, padded_tok_per_s=padded / median,
                nan_rows=int((~np.isfinite(E)).any(axis=1).sum()), dim=E.shape[1],
                max_norm_dev=float(np.nanmax(np.abs(norms - 1))))


def bench_single(model, tok, ids, batch_size, device, load_s):
    batches = make_batches(tok, ids, range(len(ids)), batch_size)
    torch.cuda.reset_peak_memory_stats(device)
    embed(model, batches[:1], device)                      # warmup
    vectors = embed(model, batches, device, keep=True)     # untimed sanity pass
    times = [timed(lambda: embed(model, batches, device), [device]) for _ in range(REPEATS)]
    real, padded = sum(b['real'] for b in batches), sum(b['padded'] for b in batches)
    return summarize(times, real, padded, torch.cuda.max_memory_allocated(device) / 2 ** 30,
                     load_s, vectors, len(ids))


def bench_dual(models, tok, ids, batch_size, load_s):
    devices = ['cuda:0', 'cuda:1']
    order = sorted(range(len(ids)), key=lambda i: -len(ids[i]))
    shards = [make_batches(tok, ids, order[k::2], batch_size) for k in range(2)]  # balanced by length
    for d in devices:
        torch.cuda.reset_peak_memory_stats(d)

    def one_pass(keep=False):
        with ThreadPoolExecutor(2) as pool:
            return list(pool.map(lambda k: embed(models[k], shards[k], devices[k], keep=keep), range(2)))

    vectors = {}
    for part in one_pass(keep=True):                       # untimed sanity pass
        vectors.update(part)
    times = [timed(one_pass, devices) for _ in range(REPEATS)]
    real = sum(b['real'] for s in shards for b in s)
    padded = sum(b['padded'] for s in shards for b in s)
    peak = max(torch.cuda.max_memory_allocated(d) for d in devices) / 2 ** 30
    return summarize(times, real, padded, peak, load_s, vectors, len(ids))


results, loads = [], []


def record(repo, config, batch_size, load_s, fn):
    """Run one benchmark cell; an OOM is recorded as a row and the run continues."""
    try:
        row, status = fn(), 'ok'
    except torch.cuda.OutOfMemoryError:
        row, status = {}, 'OOM'
        print(f'{repo} {config} batch {batch_size}: out of memory')
    gc.collect()
    torch.cuda.empty_cache()
    results.append(dict(model=repo, config=config, batch_size=batch_size, **{**dict(status=status), **row}))


bench = pd.read_parquet(f'{WORK}/bench_chunks.parquet')
texts = bench.text.tolist()
query_text = get_detailed_instruct(TASK, 'Paracetamol liều dùng cho trẻ em là bao nhiêu?')
for repo in MODELS:
    src = MODEL_SOURCE[repo]
    tok, model, load_s = load(src, 'cuda:0')
    loads.append(dict(model=repo, device='cuda:0', load_s=round(load_s, 1)))
    print(f'{repo} loaded on cuda:0 in {load_s:.1f} s')
    ids = count_tokens(tok, texts)
    qv = embed(model, make_batches(tok, count_tokens(tok, [query_text]), range(1), 1), 'cuda:0', keep=True)[0]
    print('query example:', repr(query_text[:60]), '-> vector', qv.shape)
    for batch_size in BATCH_SIZES:
        record(repo, '1 GPU', batch_size, load_s, lambda: bench_single(model, tok, ids, batch_size, 'cuda:0', load_s))
    if N_GPU >= 2:
        _, model1, load1 = load(src, 'cuda:1')
        loads.append(dict(model=repo, device='cuda:1', load_s=round(load1, 1)))
        for batch_size in BATCH_SIZES:
            record(repo, '2 GPU', batch_size, max(load_s, load1),
                   lambda: bench_dual([model, model1], tok, ids, batch_size, max(load_s, load1)))
        del model1
    else:
        print('single GPU: 2-GPU run skipped')
    del model
    gc.collect()
    torch.cuda.empty_cache()


## 6. Results and full-corpus estimate

`hours = corpus tokens / real tokens per second of the best batch size`. Corpus tokens = chunk count per language x mean tokens per chunk (section 4).
The 2 GPU row is wall-clock time for both GPUs together. The Kaggle GPU quota is about 30 h per week (**to verify** on the account page).


In [ ]:
res = pd.DataFrame(results)
cols = ['model', 'config', 'batch_size', 'status', 'load_s', 'peak_vram_gb', 'median_s', 'chunks_per_s',
        'real_tok_per_s', 'padded_tok_per_s', 'nan_rows', 'dim', 'max_norm_dev']
print(res.reindex(columns=cols).to_string(index=False))

estimate = []
for repo in MODELS:
    total_tokens = sum(CORPUS_CHUNKS[lang] * TOKEN_MEAN[repo][lang] for lang in LANGS)
    for config, gpus in [('1 GPU', 1), ('2 GPU', 2)]:
        ok = res[(res.model == repo) & (res.config == config) & (res.status == 'ok')]
        if ok.empty:
            continue
        best = ok.loc[ok.real_tok_per_s.idxmax()]
        hours = total_tokens / best.real_tok_per_s / 3600
        estimate.append(dict(model=repo, config=config, best_batch=int(best.batch_size),
                             real_tok_per_s=round(best.real_tok_per_s), corpus_chunks=sum(CORPUS_CHUNKS.values()),
                             corpus_tokens_bn=round(total_tokens / 1e9, 2), hours=round(hours, 1),
                             gpu_hours=round(hours * gpus, 1)))
estimate = pd.DataFrame(estimate)
print()
print(estimate.to_string(index=False))
print('Kaggle GPU quota ~30 h/week: TO VERIFY. Compare gpu_hours with it.')

res.to_csv(f'{WORK}/embed_benchmark.csv', index=False)
with open(f'{WORK}/embed_benchmark.json', 'w', encoding='utf-8') as file:
    json.dump(dict(gpu=GPU_NAME, n_gpu=N_GPU, dtype=str(DTYPE), max_length=MAX_LENGTH, repeats=REPEATS,
                   chunks=len(bench), chunks_per_lang=bench.lang.value_counts().to_dict(),
                   torch=torch.__version__, transformers=md.version('transformers'),
                   loads=loads, token_stats=token_rows, results=results,
                   estimate=estimate.to_dict(orient='records')),
              file, ensure_ascii=False, indent=2, default=lambda o: o.item())
print('saved', f'{WORK}/embed_benchmark.json', 'and .csv')


## Warning

100 chunks is a small sample. Throughput is indicative only, about +/-30 %. Use it to pick the batch size and to estimate GPU hours, not as a final speed measurement.
The EN and ZH token means come from the sample pool (section 4), not from the 100 chunks.
